# Pipeline de classification des maladies du riz

Exécution et évaluation du projet sur Google Colab.

---
## Guide : configuration des clés dans Google Colab (secrets)

### 1. GitHub (GITHUB_TOKEN)
- GitHub.com -> Settings -> Developer Settings -> Personal access tokens (classic) -> Generate new token.
- Cochez 'repo' et copiez le token.

### 2. Kaggle (KAGGLE_USERNAME et KAGGLE_KEY)
- Kaggle.com -> Profil -> Settings -> API -> Create New Token.
- Dans kaggle.json : recopiez les valeurs de 'username' et 'key'.

### 3. Ajouter les secrets dans Colab
- Barre latérale Colab -> Icône de clé (Secrets).
- Ajoutez les 3 variables : GITHUB_TOKEN, KAGGLE_USERNAME, KAGGLE_KEY.
- Activez l'accès ('Notebook access') pour chaque variable.

---
## 1. Cloner le dépôt et basculer sur la branche develop

In [ ]:
# Importation du module os pour les opérations sur les dossiers et fichiers
import os
# Importation du module userdata de Colab pour lire les clés secrètes
from google.colab import userdata

# Récupération du jeton d'accès GitHub depuis les Secrets Colab
github_token = userdata.get('GITHUB_TOKEN')
# Nom du répertoire de destination
repo_name = "rice-disease-rna-cv"
# Construction du chemin d'accès authentifié au dépôt distant
repo_url = f"https://{github_token}@github.com/rna-cv-m1/rice-disease-rna-cv.git"

# Clonage initial si le dossier du projet n'existe pas localement
if not os.path.exists(repo_name):
    !git clone {repo_url}
else:
    print("Dépôt déjà présent.")

# Positionnement dans le dossier du projet
%cd {repo_name}
# Basculement sur la branche develop et synchronisation
!git checkout develop
!git pull origin develop

---
## 2. Installer les dépendances

In [ ]:
# Installation silencieuse des dépendances Python requises
!pip install -q -r requirements.txt
# Installation des bibliothèques nécessaires pour le téléchargement et l'affichage des graphiques
!pip install -q kaggle matplotlib seaborn

---
## 3. Télécharger le dataset Kaggle dans data/raw/

In [ ]:
# Configuration de la variable d'environnement avec le nom d'utilisateur Kaggle
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# Configuration de la clé API Kaggle dans les variables d'environnement
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Dossier cible pour la sauvegarde des images brutes
raw_dir = "data/raw"
os.makedirs(raw_dir, exist_ok=True)

# Téléchargement automatique depuis Kaggle si le répertoire est vide
if len(os.listdir(raw_dir)) == 0:
    !kaggle datasets download -d anshulm257/rice-disease-dataset -p {raw_dir} --unzip
    print("Téléchargement terminé.")
else:
    print("Dataset déjà présent.")

---
## 4. Structure des données brutes (data/raw/)

In [ ]:
# Gestion des chemins sous forme d'objets Path
from pathlib import Path

print("=== Arborescence de data/raw/ ===")
raw_path = Path("data/raw")
# Parcours récursif des sous-dossiers pour compter le nombre d'images brutes par classe
for root, dirs, files in os.walk(raw_path):
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        rel = os.path.relpath(root, raw_path)
        print(f"  {rel:<35} | {len(imgs)} images")

---
## 5. Analyse exploratoire des données (EDA)

In [ ]:
# Exécution du module d'analyse statistique et colorimétrique (sans emojis)
!python -m src.dataset.eda

---
## 6. Prétraitement et transformation des images (data/processed/)

In [ ]:
# Parcours récursif, rognage au carré central (Center Crop) et redimensionnement à 224x224 px
!python -m src.dataset.transform

---
## 7. Comparaison avant vs après (taille sur disque)

In [ ]:
# Définition de la fonction de calcul de la taille totale d'un dossier en Mo
def get_dir_size_mb(path):
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.isfile(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)

# Calcul de la taille avant transformation
raw_mb = get_dir_size_mb("data/raw")
# Calcul de la taille après transformation et compression JPEG
proc_mb = get_dir_size_mb("data/processed")
# Calcul du pourcentage d'économie d'espace disque
gain = ((raw_mb - proc_mb) / raw_mb) * 100 if raw_mb > 0 else 0

# Affichage des métriques de stockage
print(f"Taille brute   (data/raw)       : {raw_mb:.2f} Mo")
print(f"Taille traitée (data/processed) : {proc_mb:.2f} Mo")
print(f"Gain de compression             : -{gain:.1f} %")

---
## 8. Entraîner tous les modèles et tracer les courbes d'apprentissage

In [ ]:
# Importation du module de graphiques matplotlib
import matplotlib.pyplot as plt
# Importation de la fonction d'entraînement de modèle PyTorch
from src.nn.trainer import entrainer
# Importation de la liste des 4 architectures supportées
from src.config import ARCHITECTURES_DISPONIBLES

# Dictionnaire de stockage de l'historique d'entraînement de chaque modèle
historiques = {}
# Nombre d'époques d'apprentissage
epochs = 10

# Entraînement séquentiel de chaque architecture
for arch in ARCHITECTURES_DISPONIBLES:
    print(f"\n>>> Entraînement : {arch} <<<")
    hist = entrainer(epochs=epochs, architecture=arch)
    historiques[arch] = hist

### 8a. Courbes de perte (train loss vs val loss)

In [ ]:
# Création d'une grille de graphiques 2x2 pour visualiser la perte de chaque modèle
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Tracé des courbes de train loss et val loss par modèle
for idx, (arch, hist) in enumerate(historiques.items()):
    ax = axes[idx]
    ep = range(1, len(hist['train_loss']) + 1)
    ax.plot(ep, hist['train_loss'], label='train loss', color='#1976D2')
    ax.plot(ep, hist['val_loss'], label='val loss', color='#D32F2F')
    ax.set_title(f'{arch}', fontsize=12)
    ax.set_xlabel('Époques')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Courbes de perte (train vs validation) par architecture', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 8b. Courbes de précision de validation (accuracy)

In [ ]:
# Initialisation du graphique d'évolution de la précision
plt.figure(figsize=(10, 5))
for arch, hist in historiques.items():
    ep = range(1, len(hist['val_acc']) + 1)
    plt.plot(ep, hist['val_acc'], marker='o', label=arch)

plt.title('Évolution de la précision de validation (accuracy) par architecture', fontsize=13, fontweight='bold')
plt.xlabel('Époques')
plt.ylabel('Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

---
## 9. Évaluation du meilleur modèle : rapport de classification

In [ ]:
# Importation de NumPy et Seaborn pour l'évaluation statistique
import numpy as np
import seaborn as sns
# Importation du module d'évaluation de modèles
from src.nn.evaluator import evaluer

# Calcul et affichage du rapport d'évaluation complet
res = evaluer()

### 9a. Matrice de confusion (heatmap)

In [ ]:
# Disposition à 2 colonnes pour comparer la matrice brute et la matrice normalisée en %
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Tracé 1 : Heatmap de la matrice de confusion brute (nombre d'échantillons)
sns.heatmap(res['matrix'], annot=True, fmt='d', cmap='Greens',
            xticklabels=res['classes'], yticklabels=res['classes'], ax=ax1)
ax1.set_title('Matrice de confusion (nombre)', fontweight='bold')
ax1.set_xlabel('Prédictions')
ax1.set_ylabel('Vraies classes')

# Calcul et tracé 2 : Heatmap de la matrice de confusion normalisée (pourcentages par ligne)
matrix_norm = res['matrix'].astype(float) / res['matrix'].sum(axis=1, keepdims=True)
sns.heatmap(matrix_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=res['classes'], yticklabels=res['classes'], ax=ax2)
ax2.set_title('Matrice de confusion normalisée (%)', fontweight='bold')
ax2.set_xlabel('Prédictions')
ax2.set_ylabel('Vraies classes')

plt.suptitle(f"Modèle : {res['architecture']}", fontsize=14)
plt.tight_layout()
plt.show()

### 9b. Precision, recall et F1-score par classe (bar chart)

In [ ]:
# Importation de pandas pour la mise en forme sous forme de DataFrame
import pandas as pd

classes = res['classes']
rd = res['report_dict']

# Structuration des métriques par classe dans un DataFrame
df_metrics = pd.DataFrame({
    'Classe': classes,
    'Precision': [rd[c]['precision'] * 100 for c in classes],
    'Recall': [rd[c]['recall'] * 100 for c in classes],
    'F1-score': [rd[c]['f1-score'] * 100 for c in classes],
    'Support': [rd[c]['support'] for c in classes]
})

# Positions et grandeurs pour le tracé des barres groupées
x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
# Affichage des barres Precision (vert), Recall (bleu) et F1-score (rose)
ax.bar(x - width, df_metrics['Precision'], width, label='Precision', color='#2E7D32')
ax.bar(x, df_metrics['Recall'], width, label='Recall', color='#1976D2')
ax.bar(x + width, df_metrics['F1-score'], width, label='F1-score', color='#D81B60')

ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=30, ha='right')
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 105)
ax.set_title('Precision, recall et F1-score par classe', fontweight='bold')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Affichage texte des moyennes globales macro
print(f"Accuracy globale    : {rd['accuracy']*100:.2f} %")
print(f"Macro avg precision : {rd['macro avg']['precision']*100:.2f} %")
print(f"Macro avg recall    : {rd['macro avg']['recall']*100:.2f} %")
print(f"Macro avg F1-score  : {rd['macro avg']['f1-score']*100:.2f} %")

### 9c. Support par classe (nombre d'échantillons de validation)

In [ ]:
# Visualisation sous forme de barres horizontales du volume d'images de validation par maladie
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2E7D32' if s == max(df_metrics['Support']) else '#90CAF9' for s in df_metrics['Support']]
ax.barh(df_metrics['Classe'], df_metrics['Support'], color=colors)
ax.set_xlabel('Nombre d\'échantillons')
ax.set_title('Support (nombre d\'images de validation) par classe', fontweight='bold')
for i, v in enumerate(df_metrics['Support']):
    ax.text(v + 2, i, str(int(v)), va='center')
plt.tight_layout()
plt.show()

---
## 10. Benchmark comparatif de tous les modèles

In [ ]:
# Exécution du benchmark de toutes les architectures entraînées
from src.nn.compare import comparer_modeles

bench = comparer_modeles()

# Extraction des métriques pour les histogrammes comparatifs
archs = list(bench.keys())
accs = [bench[a]['Précision (%)'] for a in archs]
times = [bench[a]["Temps d'inférence (ms/img)"] for a in archs]
sizes = [bench[a]['Taille (Mo)'] for a in archs]

# Tracé des 3 histogrammes côte-à-côte (Accuracy, Temps d'inférence, Taille sur disque)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

ax1.bar(archs, accs, color='#2E7D32')
ax1.set_title('Accuracy (%)')
ax1.set_ylim(min(accs) - 5, 100)
for i, v in enumerate(accs):
    ax1.text(i, v + 0.3, f'{v}%', ha='center', fontweight='bold')

ax2.bar(archs, times, color='#1976D2')
ax2.set_title('Temps d\'inférence (ms/img)')
for i, v in enumerate(times):
    ax2.text(i, v + 0.05, f'{v}', ha='center', fontweight='bold')

ax3.bar(archs, sizes, color='#D81B60')
ax3.set_title('Taille du fichier .pth (Mo)')
for i, v in enumerate(sizes):
    ax3.text(i, v + 0.5, f'{v}', ha='center', fontweight='bold')

plt.suptitle('Benchmark comparatif des architectures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Radar chart : synthèse multi-critères par modèle

In [ ]:
# Tracé d'un graphique radar polaire pour synthétiser les performances multi-critères
from math import pi

# Normalisation des valeurs pour les 3 axes du radar (0 à 1)
max_time = max(times) if max(times) > 0 else 1
max_size = max(sizes) if max(sizes) > 0 else 1

categories = ['Accuracy', 'Vitesse (inversée)', 'Légèreté (inversée)']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = ['#2E7D32', '#1976D2', '#D81B60', '#FF8F00']

# Superposition du polygone de chaque modèle
for idx, arch in enumerate(archs):
    vals = [
        accs[idx] / 100.0,
        1 - (times[idx] / max_time),
        1 - (sizes[idx] / max_size)
    ]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=arch, color=colors[idx % len(colors)])
    ax.fill(angles, vals, alpha=0.1, color=colors[idx % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 1)
ax.set_title('Synthèse multi-critères par modèle', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

---
## 12. Lancement de l'interface web Streamlit (tunnel Cloudflare)

In [ ]:
# Téléchargement du binaire Cloudflare Tunnel
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
# Attribution du droit d'exécution
!chmod +x cloudflared

import subprocess, time
# Lancement de l'application Streamlit en tâche de fond
streamlit_proc = subprocess.Popen([
    'streamlit', 'run', 'src/ui/app.py',
    '--server.port', '8501',
    '--server.headless', 'true',
    '--server.enableCORS', 'false'
])
# Temps d'attente pour le démarrage complet de Streamlit
time.sleep(3)

# Ouverture du tunnel et affichage de l'adresse HTTPS d'accès public
print("URL publique :")
!./cloudflared tunnel --url http://localhost:8501